In [7]:
# Cài đặt ép phiên bản 0.10.14 hoặc 0.10.9
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.14 pandas scikit-learn tensorflow opencv-python

Found existing installation: mediapipe 0.10.33
Uninstalling mediapipe-0.10.33:
  Successfully uninstalled mediapipe-0.10.33
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 35.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.2

In [2]:
import kagglehub
import os

print("Đang tải dataset...")
dataset_path = kagglehub.dataset_download("akashrayhan/yoga-82")
print("Đã tải xong! Đường dẫn:", dataset_path)

# (Tùy chọn) In ra danh sách các thư mục (các dáng yoga) để kiểm tra
# LƯU Ý: Nếu cấu trúc file của dataset nằm trong thư mục con, bạn cần trỏ đúng path
# Giả sử các ảnh nằm trong các thư mục con mang tên dáng Yoga
print(os.listdir(dataset_path)[:10]) # Xem thử 10 dáng đầu tiên

Đang tải dataset...
Using Colab cache for faster access to the 'yoga-82' dataset.
Đã tải xong! Đường dẫn: /kaggle/input/yoga-82
['valid', 'test', 'train']


In [5]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
import csv
from pathlib import Path

# 1. Khởi tạo MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

csv_file = 'yoga_keypoints.csv'
count = 0

# 2. Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(csv_file):
    os.remove(csv_file)

print(f"Đang quét dữ liệu tại: {dataset_path}")

with open(csv_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    # Header
    header = ['label']
    for i in range(33):
        header.extend([f'x{i}', f'y{i}', f'z{i}', f'v{i}'])
    writer.writerow(header)

    # Dùng rglob để tìm TẤT CẢ ảnh trong mọi thư mục con
    image_paths = list(Path(dataset_path).rglob("*.jpg")) + \
                  list(Path(dataset_path).rglob("*.jpeg")) + \
                  list(Path(dataset_path).rglob("*.png"))

    if len(image_paths) == 0:
        print("❌ KHÔNG TÌM THẤY ẢNH NÀO! Kiểm tra lại dataset_path.")
    else:
        print(f"Tìm thấy {len(image_paths)} ảnh. Đang trích xuất tọa độ...")

    for img_path in image_paths:
        # Lấy tên thư mục chứa ảnh làm Label (Tên dáng Yoga)
        label = img_path.parent.name

        image = cv2.imread(str(img_path))
        if image is None: continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        if results.pose_landmarks:
            row = [label]
            for landmark in results.pose_landmarks.landmark:
                row.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
            writer.writerow(row)
            count += 1

            # In tiến độ mỗi 100 ảnh
            if count % 100 == 0:
                print(f"✅ Đã xử lý thành công {count} ảnh...")

print(f"\n➡️ HOÀN THÀNH! Đã lưu {count} mẫu vào {csv_file}")

Đang quét dữ liệu tại: /kaggle/input/yoga-82
Tìm thấy 16767 ảnh. Đang trích xuất tọa độ...


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ Đã xử lý thành công 100 ảnh...
✅ Đã xử lý thành công 200 ảnh...
✅ Đã xử lý thành công 300 ảnh...
✅ Đã xử lý thành công 400 ảnh...
✅ Đã xử lý thành công 500 ảnh...
✅ Đã xử lý thành công 600 ảnh...
✅ Đã xử lý thành công 700 ảnh...
✅ Đã xử lý thành công 800 ảnh...
✅ Đã xử lý thành công 900 ảnh...
✅ Đã xử lý thành công 1000 ảnh...
✅ Đã xử lý thành công 1100 ảnh...
✅ Đã xử lý thành công 1200 ảnh...
✅ Đã xử lý thành công 1300 ảnh...
✅ Đã xử lý thành công 1400 ảnh...
✅ Đã xử lý thành công 1500 ảnh...
✅ Đã xử lý thành công 1600 ảnh...
✅ Đã xử lý thành công 1700 ảnh...
✅ Đã xử lý thành công 1800 ảnh...
✅ Đã xử lý thành công 1900 ảnh...
✅ Đã xử lý thành công 2000 ảnh...
✅ Đã xử lý thành công 2100 ảnh...
✅ Đã xử lý thành công 2200 ảnh...
✅ Đã xử lý thành công 2300 ảnh...
✅ Đã xử lý thành công 2400 ảnh...
✅ Đã xử lý thành công 2500 ảnh...
✅ Đã xử lý thành công 2600 ảnh...
✅ Đã xử lý thành công 2700 ảnh...
✅ Đã xử lý thành công 2800 ảnh...
✅ Đã xử lý thành công 2900 ảnh...
✅ Đã xử lý thành công 3

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pickle

# =========================
# 1) Load data
# =========================
df = pd.read_csv('yoga_keypoints.csv')
print(f"Kich thuoc du lieu goc: {df.shape}")

X_raw = df.drop('label', axis=1).astype('float32').values
y_labels = df['label'].values

# =========================
# 2) Feature engineering
# - Hip-center normalization
# - Torso scale normalization
# - Joint-angle features
# - Bone-length ratio features
# =========================
ANGLE_TRIPLETS = [
    (11, 13, 15),  # left elbow
    (12, 14, 16),  # right elbow
    (13, 11, 23),  # left shoulder
    (14, 12, 24),  # right shoulder
    (11, 23, 25),  # left hip
    (12, 24, 26),  # right hip
    (23, 25, 27),  # left knee
    (24, 26, 28),  # right knee
    (25, 27, 31),  # left ankle
    (26, 28, 32),  # right ankle
]


def calc_angle(a, b, c):
    ba = a - b
    bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    cosine = np.dot(ba, bc) / denom
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.degrees(np.arccos(cosine)) / 180.0  # normalize to [0, 1]


def safe_ratio(num, den):
    return num / (den + 1e-8)


def normalize_pose(landmarks_33x4):
    xyz = landmarks_33x4[:, :3]
    visibility = landmarks_33x4[:, 3]

    hip_center = (xyz[23] + xyz[24]) / 2.0
    shoulder_center = (xyz[11] + xyz[12]) / 2.0
    torso_scale = np.linalg.norm(shoulder_center - hip_center)
    if torso_scale < 1e-6:
        torso_scale = 1.0

    xyz_norm = (xyz - hip_center) / torso_scale
    return xyz_norm, visibility


def build_feature_matrix(raw_values):
    feature_rows = []

    for row in raw_values:
        lm = row.reshape(33, 4)
        xyz_norm, visibility = normalize_pose(lm)

        angles = [calc_angle(xyz_norm[a], xyz_norm[b], xyz_norm[c]) for a, b, c in ANGLE_TRIPLETS]

        # Ratios reduce dependence on body size
        l_upper_arm = np.linalg.norm(xyz_norm[11] - xyz_norm[13])
        l_fore_arm = np.linalg.norm(xyz_norm[13] - xyz_norm[15])
        r_upper_arm = np.linalg.norm(xyz_norm[12] - xyz_norm[14])
        r_fore_arm = np.linalg.norm(xyz_norm[14] - xyz_norm[16])
        l_upper_leg = np.linalg.norm(xyz_norm[23] - xyz_norm[25])
        l_lower_leg = np.linalg.norm(xyz_norm[25] - xyz_norm[27])
        r_upper_leg = np.linalg.norm(xyz_norm[24] - xyz_norm[26])
        r_lower_leg = np.linalg.norm(xyz_norm[26] - xyz_norm[28])

        ratios = [
            safe_ratio(l_fore_arm, l_upper_arm),
            safe_ratio(r_fore_arm, r_upper_arm),
            safe_ratio(l_lower_leg, l_upper_leg),
            safe_ratio(r_lower_leg, r_upper_leg),
        ]

        feature_row = np.concatenate([
            xyz_norm.flatten(),   # 99 dims
            visibility,           # 33 dims
            np.array(angles),     # 10 dims
            np.array(ratios),     # 4 dims
        ]).astype(np.float32)

        feature_rows.append(feature_row)

    return np.vstack(feature_rows)


X_features = build_feature_matrix(X_raw)
print(f"Kich thuoc feature sau chuan hoa: {X_features.shape}")

# =========================
# 3) Encode labels + split
# =========================
label_encoder = LabelEncoder()
y_idx = label_encoder.fit_transform(y_labels)
num_classes = len(np.unique(y_idx))

X_train, X_temp, y_train_idx, y_temp_idx = train_test_split(
    X_features, y_idx, test_size=0.30, random_state=42, stratify=y_idx
)
X_val, X_test, y_val_idx, y_test_idx = train_test_split(
    X_temp, y_temp_idx, test_size=0.50, random_state=42, stratify=y_temp_idx
)

print(f"Train/Val/Test: {len(X_train)}/{len(X_val)}/{len(X_test)}")

# Scale only from train set to avoid data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = tf.keras.utils.to_categorical(y_train_idx, num_classes=num_classes)
y_val = tf.keras.utils.to_categorical(y_val_idx, num_classes=num_classes)
y_test = tf.keras.utils.to_categorical(y_test_idx, num_classes=num_classes)

# Handle class imbalance
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_idx),
    y=y_train_idx
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}

# =========================
# 4) Build and train model
# =========================
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.35),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
    ],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_yoga_pose_model.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
]

print('Bat dau huan luyen...')
history = model.fit(
    X_train, y_train,
    epochs=120,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

# =========================
# 5) Evaluate
# =========================
loss, acc, top3 = model.evaluate(X_test, y_test, verbose=0)
y_pred_prob = model.predict(X_test, verbose=0)
y_pred_idx = np.argmax(y_pred_prob, axis=1)
macro_f1 = f1_score(y_test_idx, y_pred_idx, average='macro')

print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test Top-3 Accuracy: {top3 * 100:.2f}%")
print(f"Test Macro F1: {macro_f1:.4f}")

print('\nClassification report:')
print(classification_report(y_test_idx, y_pred_idx, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test_idx, y_pred_idx)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
cm_df.to_csv('confusion_matrix.csv', encoding='utf-8-sig')
print('Da luu confusion_matrix.csv')

# =========================
# 6) Save model + preprocessing artifacts
# =========================
model.save('yoga_pose_model.keras')

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

preprocessing_artifacts = {
    'feature_version': 'v2_hip_center_torso_scale_angles_ratios',
    'scaler': scaler,
    'label_encoder': label_encoder,
    'angle_triplets': ANGLE_TRIPLETS,
    'input_dim': int(X_train.shape[1]),
}

with open('pose_preprocessing.pkl', 'wb') as f:
    pickle.dump(preprocessing_artifacts, f)

history_df = pd.DataFrame(history.history)
history_df.to_csv('training_history.csv', index=False)

print('Da luu: yoga_pose_model.keras, label_encoder.pkl, pose_preprocessing.pkl, training_history.csv')

Bắt đầu huấn luyện...
Epoch 1/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.0924 - loss: 3.9157 - val_accuracy: 0.2486 - val_loss: 3.1486
Epoch 2/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.2352 - loss: 2.9608 - val_accuracy: 0.3787 - val_loss: 2.4062
Epoch 3/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3232 - loss: 2.4994 - val_accuracy: 0.4857 - val_loss: 1.9907
Epoch 4/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3946 - loss: 2.1926 - val_accuracy: 0.5483 - val_loss: 1.7604
Epoch 5/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4504 - loss: 2.0238 - val_accuracy: 0.5651 - val_loss: 1.6338
Epoch 6/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4688 - loss: 1.9115 - val_accuracy: 0.5930 - val_loss: 1.5386
Epoch 7/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4992 - loss: 1.8194 - val_accuracy: 0.6347 - val_loss: 1.4052
Epoch 8/50
372/372 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5188 - loss: 1.

Độ chính xác trên tập test: 77.67%
Đã lưu model tại yoga_pose_model.h5


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import pickle

# Load trained artifacts
inference_model = tf.keras.models.load_model('yoga_pose_model.keras')
with open('pose_preprocessing.pkl', 'rb') as f:
    inference_artifacts = pickle.load(f)

inference_scaler = inference_artifacts['scaler']
inference_label_encoder = inference_artifacts['label_encoder']
inference_angle_triplets = inference_artifacts['angle_triplets']


def infer_calc_angle(a, b, c):
    ba = a - b
    bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    cosine = np.dot(ba, bc) / denom
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.degrees(np.arccos(cosine)) / 180.0


def infer_safe_ratio(num, den):
    return num / (den + 1e-8)


def infer_normalize_pose(landmarks_33x4):
    xyz = landmarks_33x4[:, :3]
    visibility = landmarks_33x4[:, 3]

    hip_center = (xyz[23] + xyz[24]) / 2.0
    shoulder_center = (xyz[11] + xyz[12]) / 2.0
    torso_scale = np.linalg.norm(shoulder_center - hip_center)
    if torso_scale < 1e-6:
        torso_scale = 1.0

    xyz_norm = (xyz - hip_center) / torso_scale
    return xyz_norm, visibility


def infer_build_feature(landmarks_33x4):
    xyz_norm, visibility = infer_normalize_pose(landmarks_33x4)

    angles = [
        infer_calc_angle(xyz_norm[a], xyz_norm[b], xyz_norm[c])
        for a, b, c in inference_angle_triplets
    ]

    l_upper_arm = np.linalg.norm(xyz_norm[11] - xyz_norm[13])
    l_fore_arm = np.linalg.norm(xyz_norm[13] - xyz_norm[15])
    r_upper_arm = np.linalg.norm(xyz_norm[12] - xyz_norm[14])
    r_fore_arm = np.linalg.norm(xyz_norm[14] - xyz_norm[16])
    l_upper_leg = np.linalg.norm(xyz_norm[23] - xyz_norm[25])
    l_lower_leg = np.linalg.norm(xyz_norm[25] - xyz_norm[27])
    r_upper_leg = np.linalg.norm(xyz_norm[24] - xyz_norm[26])
    r_lower_leg = np.linalg.norm(xyz_norm[26] - xyz_norm[28])

    ratios = [
        infer_safe_ratio(l_fore_arm, l_upper_arm),
        infer_safe_ratio(r_fore_arm, r_upper_arm),
        infer_safe_ratio(l_lower_leg, l_upper_leg),
        infer_safe_ratio(r_lower_leg, r_upper_leg),
    ]

    feature = np.concatenate([
        xyz_norm.flatten(),
        visibility,
        np.array(angles),
        np.array(ratios),
    ]).astype(np.float32)

    feature = inference_scaler.transform(feature.reshape(1, -1))
    return feature


def predict_pose_from_image(image_path, target_pose=None, min_detection_confidence=0.5):
    mp_pose = mp.solutions.pose
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f'Khong doc duoc anh: {image_path}')

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with mp_pose.Pose(
        static_image_mode=True,
        min_detection_confidence=min_detection_confidence,
    ) as pose:
        results = pose.process(image_rgb)

    if not results.pose_landmarks:
        raise ValueError('Khong phat hien duoc pose landmarks trong anh.')

    lm = np.array(
        [[p.x, p.y, p.z, p.visibility] for p in results.pose_landmarks.landmark],
        dtype=np.float32,
    )

    x_input = infer_build_feature(lm)
    probs = inference_model.predict(x_input, verbose=0)[0]

    top_idx = int(np.argmax(probs))
    top_label = inference_label_encoder.inverse_transform([top_idx])[0]
    top_conf = float(probs[top_idx])

    top3_idx = np.argsort(probs)[-3:][::-1]
    top3 = [
        {
            'label': inference_label_encoder.inverse_transform([int(i)])[0],
            'confidence': float(probs[int(i)]),
        }
        for i in top3_idx
    ]

    result = {
        'predicted_label': top_label,
        'confidence': top_conf,
        'top3': top3,
    }

    if target_pose is not None:
        result['target_pose'] = target_pose
        result['is_match'] = (top_label == target_pose)

    return result


# Example usage:
# result = predict_pose_from_image('path_to_image.jpg', target_pose='Tree')
# print(result)